# Comprobación de inferencia de RoBERTuito

Este cuaderno compara las predicciones del test de semilla 42 con el modelo exportado que se usó para SHAP. **No entrena, no optimiza y no sustituye resultados existentes.**

Ejecutar todas las celdas en el equipo con GPU CUDA y el entorno donde se ejecutó `04c_RoBERTuito.ipynb`. No es necesario volver a ejecutar ese notebook.

La auditoría inicial funciona sin PyTorch. Las predicciones necesitan PyTorch, Transformers y Accelerate del entorno original. La ejecución GPU se detiene si CUDA/BF16 no están disponibles. Cada ejecución guarda sus archivos en una carpeta nueva.

El cuaderno comprueba: textos/etiquetas y archivos guardados; predicción manual FP32 con lotes de 64 y 16; padding a múltiplos de ocho; precisión BF16; y predicción mediante Trainer BF16. Los dos modos manuales con lote 16 y padding 8 aíslan el cambio de precisión. No se utiliza el test para elegir hiperparámetros ni para ajustar el modelo.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# Solo cambiar si el proyecto está en otra carpeta.
candidatas = [Path.cwd(), Path.cwd().parent]
RAIZ_PROYECTO = next((p for p in candidatas if (p / "data" / "resultados_robertuito_grouped_5x3_tpe20").is_dir()), None)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError("Abre este notebook desde la carpeta notebooks o la raíz de DeteccionDeSarcasmo.")

marca = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SALIDA = RAIZ_PROYECTO / "data" / "diagnostico_inferencia_robertuito" / marca
SALIDA.mkdir(parents=True, exist_ok=False)
print("Resultados nuevos en:", SALIDA.resolve())


## 1. Funciones de comprobación

Definir estas funciones no carga los modelos ni ejecuta entrenamiento.

In [ ]:
"""Diagnóstico de inferencia con pesos fijos. No entrena ni sobrescribe resultados previos."""
from pathlib import Path
from datetime import datetime, timezone
import csv
import gc
import hashlib
import inspect
import json
import pickletools
import zipfile


def huella(ruta):
    digest = hashlib.sha256()
    with Path(ruta).open('rb') as f:
        for bloque in iter(lambda: f.read(8 * 1024 * 1024), b''):
            digest.update(bloque)
    return digest.hexdigest()


def argumentos_guardados(ruta):
    """Lee primitivas del archivo de argumentos sin ejecutar su contenido pickle."""
    claves = {'bf16', 'fp16', 'bf16_full_eval', 'fp16_full_eval',
              'per_device_eval_batch_size', 'tf32', 'seed'}
    with zipfile.ZipFile(ruta) as z:
        entrada = next(n for n in z.namelist() if n.endswith('/data.pkl'))
        operaciones = list(pickletools.genops(z.read(entrada)))
    resultado = {}
    for i, (_, clave, _) in enumerate(operaciones):
        if not isinstance(clave, str) or clave not in claves:
            continue
        j = i + 1
        while operaciones[j][0].name in {'BINPUT', 'LONG_BINPUT', 'MEMOIZE'}:
            j += 1
        op, valor, _ = operaciones[j]
        if op.name in {'NEWTRUE', 'NEWFALSE', 'NONE'}:
            valor = {'NEWTRUE': True, 'NEWFALSE': False, 'NONE': None}[op.name]
        elif op.name not in {'BININT', 'BININT1', 'BININT2', 'INT', 'LONG1', 'LONG4'}:
            raise ValueError(f'No se reconoce el valor primitivo de {clave}: {op.name}')
        resultado[clave] = valor
    if not claves.issubset(resultado):
        raise ValueError(f'No se pudieron leer los argumentos: {claves - resultado.keys()}')
    return resultado


def leer_csv(ruta):
    with Path(ruta).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))


def auditar_archivos(raiz_proyecto, salida):
    raiz = Path(raiz_proyecto).resolve()
    salida = Path(salida).resolve()
    salida.mkdir(parents=True, exist_ok=True)
    datos = raiz/'data'
    experimento = datos/'resultados_robertuito_grouped_5x3_tpe20'
    rutas = [experimento/'robertuito_test_probabilidades.csv',
             experimento/'robertuito_test_predicciones_modelo_exportado.csv']
    original = {(r['variante'], r['ID']): r for r in leer_csv(rutas[0]) if int(r['seed']) == 42}
    exportado = {(r['variante'], r['ID']): r for r in leer_csv(rutas[1])}
    assert len(original) == len(exportado) == 1800
    assert set(original) == set(exportado)
    informe = {'modelos': {}, 'discrepancias': []}
    for variante in ['mx', 'es', 'cu']:
        carpeta = experimento/'modelo_final'/variante
        meta = json.loads((carpeta/'metadatos.json').read_text(encoding='utf-8'))
        flags = argumentos_guardados(carpeta/'training_args.bin')
        assert meta['seed'] == 42 and flags['seed'] == 42
        assert flags['bf16'] is True and flags['fp16'] is False
        assert flags['per_device_eval_batch_size'] == 16
        assert not flags['bf16_full_eval'] and not flags['fp16_full_eval']
        entrada = datos/'transformers'/f'test_transformers_{variante}.csv'
        filas = leer_csv(entrada)
        assert len(filas) == 600 and len({r['ID'] for r in filas}) == 600
        for fila in filas:
            clave = (variante, fila['ID'])
            a, b = original[clave], exportado[clave]
            assert fila[meta['columna_texto']] == b['texto']
            assert int(fila['IS_IRONIC']) == int(a['y_true']) == int(b['y_true'])
            if int(a['y_pred']) != int(b['y_pred']):
                informe['discrepancias'].append({'variante': variante, 'ID': fila['ID'],
                    'clase_original': int(a['y_pred']), 'clase_exportada': int(b['y_pred']),
                    'prob_original': float(a['prob_ironia']), 'prob_exportada': float(b['prob_ironia'])})
        rutas += [entrada, carpeta/'metadatos.json', carpeta/'training_args.bin',
                  carpeta/'model.safetensors', carpeta/'tokenizer.json', carpeta/'tokenizer_config.json',
                  carpeta/'config.json']
        informe['modelos'][variante] = {'metadatos': meta, 'argumentos_originales': flags}
    informe['fuentes_sha256'] = {str(p.relative_to(raiz)): huella(p) for p in rutas}
    informe['n_discrepancias_guardadas'] = len(informe['discrepancias'])
    informe['textos_y_etiquetas_coinciden'] = True
    informe['ejecuto_inferencia'] = False
    (salida/'auditoria_archivos.json').write_text(json.dumps(informe, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Archivos verificados. Diferencias guardadas:', len(informe['discrepancias']))
    print('Las tres ejecuciones originales: BF16, batch 16; los pesos guardados son de semilla 42.')
    return informe


def comprobar_inferencia(raiz_proyecto, salida, auditoria):
    # Las dependencias se importan aquí: la auditoría de archivos funciona sin ellas.
    import numpy as np
    import pandas as pd
    import torch
    import transformers
    import accelerate
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Para contrastar la evaluación BF16 original, ejecuta esta sección en la GPU CUDA del equipo de entrenamiento. La auditoría de archivos ya se guardó; no se ha entrenado ni modificado ningún modelo.')
    raiz = Path(raiz_proyecto).resolve()
    salida = Path(salida).resolve()
    datos = raiz/'data'
    exp = datos/'resultados_robertuito_grouped_5x3_tpe20'
    torch.set_float32_matmul_precision('high')
    entorno = {'torch': torch.__version__, 'transformers': transformers.__version__,
               'accelerate': accelerate.__version__, 'numpy': np.__version__,
               'gpu': torch.cuda.get_device_name(), 'cuda': torch.version.cuda,
               'precision_matmul': torch.get_float32_matmul_precision()}
    (salida/'entorno.json').write_text(json.dumps(entorno, indent=2), encoding='utf-8')

    class DatosTokenizados(torch.utils.data.Dataset):
        def __init__(self, codificacion, etiquetas):
            self.codificacion, self.etiquetas = codificacion, etiquetas
        def __len__(self):
            return len(self.etiquetas)
        def __getitem__(self, indice):
            return {**{k: v[indice] for k, v in self.codificacion.items()},
                    'labels': int(self.etiquetas[indice])}

    class TrainerSoloPrediccion(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            # Igual ruta de logits que el entrenador original; la ponderación de la
            # pérdida no modifica los logits y no hay entrenamiento/enmascaramiento.
            outputs = model(**{k: v for k, v in inputs.items() if k != 'labels'})
            loss = torch.nn.functional.cross_entropy(outputs.logits, inputs['labels'])
            return (loss, outputs) if return_outputs else loss

    modos = [
        ('manual_fp32_lote64', 'manual', False, 64, None),
        ('manual_fp32_lote16', 'manual', False, 16, None),
        ('manual_fp32_lote16_pad8', 'manual', False, 16, 8),
        ('manual_bf16_lote16_pad8', 'manual', True, 16, 8),
        ('trainer_bf16_lote16_pad8', 'trainer', True, 16, 8),
    ]
    oficial = pd.read_csv(exp/'robertuito_test_probabilidades.csv', dtype={'ID': str})
    oficial = oficial[oficial.seed.eq(42)]
    exportado = pd.read_csv(exp/'robertuito_test_predicciones_modelo_exportado.csv', dtype={'ID': str})
    resumen, detalles, tokens_sha = [], [], {}
    for v in ['mx', 'es', 'cu']:
        carpeta = exp/'modelo_final'/v
        meta = auditoria['modelos'][v]['metadatos']
        df = pd.read_csv(datos/'transformers'/f'test_transformers_{v}.csv', dtype={'ID': str})
        ref = oficial[oficial.variante.eq(v)].set_index('ID').loc[df.ID]
        ex = exportado[exportado.variante.eq(v)].set_index('ID').loc[df.ID]
        textos = df[meta['columna_texto']].astype(str).tolist()
        tok = AutoTokenizer.from_pretrained(str(carpeta), local_files_only=True)
        enc = tok(textos, truncation=True, max_length=meta['max_length'], padding=False)
        tokens_sha[v] = hashlib.sha256(json.dumps(dict(enc), sort_keys=True).encode()).hexdigest()
        dataset = DatosTokenizados(enc, df.IS_IRONIC.to_numpy())
        for nombre, ruta, bf16, lote, multiplo in modos:
            print(v.upper(), nombre, flush=True)
            modelo = AutoModelForSequenceClassification.from_pretrained(str(carpeta), local_files_only=True).float().eval()
            entrenador = None
            try:
                if ruta == 'manual':
                    modelo.to('cuda')
                    partes = []
                    for inicio in range(0, len(textos), lote):
                        entrada = tok(textos[inicio:inicio+lote], truncation=True,
                            max_length=meta['max_length'], padding=True,
                            pad_to_multiple_of=multiplo, return_tensors='pt').to('cuda')
                        with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=bf16):
                            partes.append(modelo(**entrada).logits.float().cpu().numpy())
                    logits = np.vstack(partes)
                else:
                    args = TrainingArguments(output_dir=str(salida/'temporales'/v/nombre),
                        per_device_eval_batch_size=lote, bf16=bf16, fp16=False,
                        bf16_full_eval=False, fp16_full_eval=False,
                        seed=42, data_seed=42, report_to='none', disable_tqdm=True,
                        dataloader_num_workers=0, remove_unused_columns=False)
                    opciones = dict(model=modelo, args=args,
                        data_collator=DataCollatorWithPadding(tokenizer=tok, pad_to_multiple_of=multiplo))
                    firma = inspect.signature(Trainer.__init__).parameters
                    opciones['processing_class' if 'processing_class' in firma else 'tokenizer'] = tok
                    entrenador = TrainerSoloPrediccion(**opciones)
                    logits = entrenador.predict(dataset).predictions
                    if isinstance(logits, tuple):
                        logits = logits[0]
                z = np.asarray(logits, dtype=np.float64)
                z -= z.max(axis=1, keepdims=True)
                prob = np.exp(z); prob /= prob.sum(axis=1, keepdims=True)
                pred = prob.argmax(axis=1)
                a = np.abs(prob[:,1]-ref.prob_ironia.to_numpy())
                b = np.abs(prob[:,1]-ex.prob_ironia.to_numpy())
                versiones_iguales = all(str(entorno[k]) == str(meta['versiones'][k]) for k in ['torch','transformers','numpy'])
                resumen.append(dict(variante=v,modo=nombre,n_textos=len(df),
                    cambios_vs_original=int((pred!=ref.y_pred.to_numpy()).sum()),
                    cambios_vs_exportado=int((pred!=ex.y_pred.to_numpy()).sum()),
                    max_dif_prob_original=float(a.max()),max_dif_prob_exportada=float(b.max()),
                    versiones_principales_iguales=versiones_iguales))
                detalles.append(pd.DataFrame(dict(variante=v,modo=nombre,ID=df.ID.to_numpy(),
                    y_true=df.IS_IRONIC.to_numpy(),y_pred=pred,prob_ironia=prob[:,1],
                    pred_original=ref.y_pred.to_numpy(),prob_original=ref.prob_ironia.to_numpy(),
                    pred_exportada=ex.y_pred.to_numpy(),prob_exportada=ex.prob_ironia.to_numpy())))
                pd.DataFrame(resumen).to_csv(salida/'resumen_comparacion.csv',index=False)
                pd.concat(detalles,ignore_index=True).to_csv(salida/'comparacion_por_texto.csv',index=False)
            finally:
                del entrenador, modelo
                gc.collect()
                torch.cuda.empty_cache()
    assert all(huella(raiz/r)==h for r,h in auditoria['fuentes_sha256'].items()), 'Cambió algún archivo de entrada durante la ejecución.'
    (salida/'tokenizacion_sha256.json').write_text(json.dumps(tokens_sha,indent=2), encoding='utf-8')
    res = pd.DataFrame(resumen)
    base=res[res.modo.eq('manual_fp32_lote64')]
    bf=res[res.modo.eq('trainer_bf16_lote16_pad8')]
    control=res[res.modo.eq('manual_fp32_lote16_pad8')]
    precision=res[res.modo.eq('manual_bf16_lote16_pad8')]
    reproducen = base.cambios_vs_exportado.eq(0).all() and bf.cambios_vs_original.eq(0).all()
    todas_versiones=res.versiones_principales_iguales.all()
    notas = [
        '# Comprobación de inferencia de RoBERTuito', '',
        'No se entrenaron modelos ni se sustituyeron resultados existentes. Los archivos de entrada conservan sus huellas.', '',
        '## Resultado', '',
        'La ruta manual reproduce las clases exportadas y la ruta BF16 de Trainer reproduce las clases originales.' if reproducen else
        'Las rutas no reproducen completamente ambas referencias. La correspondencia sigue pendiente; revisar el detalle por texto y el entorno.', '',
        'Las versiones principales coinciden con los metadatos guardados.' if todas_versiones else
        'Las versiones principales difieren de los metadatos originales; conservar esta limitación y, si es necesario, repetir en el entorno original.', '',
        'La coincidencia de clases no equivale a igualdad exacta de probabilidades. Las diferencias máximas de probabilidad figuran en el resumen.', '',
        'El contraste entre manual_fp32_lote16_pad8 y manual_bf16_lote16_pad8 cambia únicamente el contexto BF16 dentro de este cuaderno; los lotes, el padding, los datos y los pesos permanecen iguales. El modo Trainer BF16 contrasta además la ruta del evaluador original. Consultar las tres filas antes de atribuir los cambios a precisión.', '',
        '## Resumen del control de precisión y Trainer', '',
        control.to_csv(index=False), precision.to_csv(index=False), bf.to_csv(index=False),
        '## Próximo paso', '',
        'Revisar resumen_comparacion.csv y comparacion_por_texto.csv. No reemplazar automáticamente las tablas ni SHAP. Si se confirma una diferencia de precisión sin cambio de pesos, documentar las rutas; si se necesita una única ruta de inferencia, regenerar solo las salidas afectadas.',
    ]
    (salida/'Informe_comprobacion.md').write_text('\n'.join(notas), encoding='utf-8')
    (salida/'estado.json').write_text(json.dumps({'completo':True,
        'clases_de_ambas_referencias_reproducidas':bool(reproducen),
        'versiones_principales_iguales':bool(todas_versiones),
        'fuentes_sin_cambios':True},indent=2), encoding='utf-8')
    print('Comprobación finalizada. Resultados separados en:',salida)
    return res


## 2. Auditoría de archivos

Verifica los 1 800 textos, las etiquetas, la semilla y los argumentos originales. Calcula huellas de los archivos para comprobar que no cambian durante el diagnóstico.

In [ ]:
AUDITORIA = auditar_archivos(RAIZ_PROYECTO, SALIDA)
print("Discrepancias guardadas:", AUDITORIA["n_discrepancias_guardadas"])

## 3. Comparación con los pesos guardados

Esta celda ejecuta únicamente predicciones. Requiere la GPU CUDA con BF16 del equipo original. Se comparan los 600 textos de cada variante, conservando todos los IDs y el orden. Las versiones y la GPU quedan registradas.

In [ ]:
RESUMEN = comprobar_inferencia(RAIZ_PROYECTO, SALIDA, AUDITORIA)
display(RESUMEN)

## 4. Qué revisar al terminar

En la carpeta indicada al principio se guardan:

- `Informe_comprobacion.md`: resultado y alcance.
- `resumen_comparacion.csv`: diferencias por variante y modo de inferencia.
- `comparacion_por_texto.csv`: probabilidades y clases de cada texto.
- `entorno.json`, `auditoria_archivos.json` y `tokenizacion_sha256.json`: trazabilidad.
- `estado.json`: indica si la ejecución se completó y si ambas referencias se reprodujeron en clase.

Las columnas `cambios_vs_original` y `cambios_vs_exportado` cuentan cambios de clase respecto de los archivos anteriores. Cero cambios de clase no implica probabilidades exactamente iguales: revisar también las diferencias máximas.

No sustituir automáticamente las tablas ni las explicaciones SHAP. Si el cuaderno reproduce las dos referencias y el control de precisión explica la diferencia, documentar ambas rutas. Si no reproduce alguna referencia o cambian las versiones, revisar el entorno antes de atribuir una causa.
